# NPY Frames Preview

Preview a few frames from motion files in `ase/data/motions/pingpong`.

Run the import bootstrap cell first so `poselib` imports correctly in Jupyter.

This notebook shows:
- file-level metadata
- selected frame quaternions (`rotation.arr`)
- selected frame root translation (`root_translation.arr`)

In [2]:
# Notebook import bootstrap for local repo modules
import sys
from pathlib import Path

# Resolve repo root whether notebook is run from repo root or from ase/data
cwd = Path.cwd().resolve()
if (cwd / "ase").exists():
    repo_root = cwd
elif (cwd.parent / "ase").exists():
    repo_root = cwd.parent
elif (cwd.parent.parent / "ase").exists():
    repo_root = cwd.parent.parent
else:
    raise RuntimeError("Could not locate repo root containing 'ase' directory.")

ase_path = repo_root / "ase"
if str(ase_path) not in sys.path:
    sys.path.insert(0, str(ase_path))

print("repo_root:", repo_root)
print("ase_path added to sys.path:", ase_path)

# Canonical import used in this notebook
from poselib.poselib.skeleton.skeleton3d import SkeletonMotion
print("poselib import OK")

2026-04-27 18:07:22,434 - INFO - logger - logger initialized


repo_root: /Users/zainaqasim/Repositories/pingpong
ase_path added to sys.path: /Users/zainaqasim/Repositories/pingpong/ase
poselib import OK


In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys

# Paths
MOTION_DIR = repo_root / "ase" / "data" / "motions" / "pingpong"
OUT_DIR = repo_root / "output" / "analysis"
OUT_DIR.mkdir(parents=True, exist_ok=True)

npy_files = sorted(MOTION_DIR.glob("*.npy"))
print(f"Found {len(npy_files)} .npy motion files in {MOTION_DIR}")
npy_files[:5]

Found 30 .npy motion files in /Users/zainaqasim/Repositories/pingpong/ase/data/motions/pingpong


[PosixPath('/Users/zainaqasim/Repositories/pingpong/ase/data/motions/pingpong/ROM_blender_pph.npy'),
 PosixPath('/Users/zainaqasim/Repositories/pingpong/ase/data/motions/pingpong/Transition_looping_FB_blender_pph.npy'),
 PosixPath('/Users/zainaqasim/Repositories/pingpong/ase/data/motions/pingpong/backhand_blender_pph.npy'),
 PosixPath('/Users/zainaqasim/Repositories/pingpong/ase/data/motions/pingpong/backhand_block_01_blender_pph.npy'),
 PosixPath('/Users/zainaqasim/Repositories/pingpong/ase/data/motions/pingpong/backhand_chop001_blender_pph.npy')]

In [4]:
# Use absolute import after bootstrap cell adjusts sys.path
from poselib.poselib.skeleton.skeleton3d import SkeletonMotion

In [5]:
def load_payload(path: Path):
    arr = np.load(path, allow_pickle=True)
    if not (isinstance(arr, np.ndarray) and arr.dtype == object and arr.size == 1):
        raise ValueError(f"Unexpected npy layout in {path}")
    payload = arr.item()
    if not isinstance(payload, dict):
        raise ValueError(f"Payload is not dict in {path}")
    return payload


def extract_arrays(payload: dict):
    # Expected SkeletonMotion layout
    rot = payload["rotation"]["arr"]               # [F, J, 4]
    root_t = payload["root_translation"]["arr"]    # [F, 3]
    gvel = payload["global_velocity"]["arr"]       # [F, J, 3]
    gavel = payload["global_angular_velocity"]["arr"]  # [F, J, 3]
    fps = float(payload.get("fps", 30.0))
    return rot, root_t, gvel, gavel, fps


def summarize_numeric(arr: np.ndarray):
    flat = arr.astype(np.float64, copy=False).reshape(-1)
    finite = flat[np.isfinite(flat)]
    if finite.size == 0:
        return {
            "min": np.nan, "max": np.nan, "mean": np.nan, "std": np.nan,
            "nan_count": int(np.isnan(flat).sum()),
            "inf_count": int(np.isinf(flat).sum()),
        }
    return {
        "min": float(np.min(finite)),
        "max": float(np.max(finite)),
        "mean": float(np.mean(finite)),
        "std": float(np.std(finite)),
        "nan_count": int(np.isnan(flat).sum()),
        "inf_count": int(np.isinf(flat).sum()),
    }

In [6]:
# 0) Quick look at raw data structure
test_file_path = sorted(npy_files)[0]
print("Using:", test_file_path)


Using: /Users/zainaqasim/Repositories/pingpong/ase/data/motions/pingpong/ROM_blender_pph.npy


In [7]:
payload = np.load(test_file_path, allow_pickle=True).item()

In [8]:
payload.keys()

odict_keys(['rotation', 'root_translation', 'global_velocity', 'global_angular_velocity', 'skeleton_tree', 'is_local', 'fps', '__name__'])

In [9]:
# Check if all files have the same keys 
all_keys = set(payload.keys())
for p in npy_files:
    payload = np.load(p, allow_pickle=True).item()
    if set(payload.keys()) != all_keys:
        raise ValueError(f"Keys mismatch in {p}")
print("All files have the same keys")


All files have the same keys


In [10]:
for key in payload.keys():
    value = payload[key]
    if isinstance(value, dict):
        print(f"Key '{key}': sub-keys = {list(value.keys())}")
    else:
        print(f"Key '{key}': not a dict (type={type(value).__name__})")

Key 'rotation': sub-keys = ['arr', 'context']
Key 'root_translation': sub-keys = ['arr', 'context']
Key 'global_velocity': sub-keys = ['arr', 'context']
Key 'global_angular_velocity': sub-keys = ['arr', 'context']
Key 'skeleton_tree': sub-keys = ['node_names', 'parent_indices', 'local_translation']
Key 'is_local': not a dict (type=bool)
Key 'fps': not a dict (type=float)
Key '__name__': not a dict (type=str)


In [ ]:
# Inspection of payload structure according to known keys/subkeys

# 'rotation': dict with sub-keys ['arr', 'context']
rotation = payload['rotation']
print("Key 'rotation': sub-keys =", list(rotation.keys()))
print("  First element of 'rotation[\"arr\"]':", rotation['arr'][0])
print("  First element of 'rotation[\"context\"]':", rotation['context'][0] if isinstance(rotation['context'], (list, np.ndarray)) else rotation['context'])

In [ ]:
# 'root_translation': dict with sub-keys ['arr', 'context']
root_translation = payload['root_translation']
print("Key 'root_translation': sub-keys =", list(root_translation.keys()))
print("  First element of 'root_translation[\"arr\"]':", root_translation['arr'][0])
print("  First element of 'root_translation[\"context\"]':", root_translation['context'][0] if isinstance(root_translation['context'], (list, np.ndarray)) else root_translation['context'])


In [ ]:
# 'global_velocity': dict with sub-keys ['arr', 'context']
global_velocity = payload['global_velocity']
print("Key 'global_velocity': sub-keys =", list(global_velocity.keys()))
print("  First element of 'global_velocity[\"arr\"]':", global_velocity['arr'][0])
print("  First element of 'global_velocity[\"context\"]':", global_velocity['context'][0] if isinstance(global_velocity['context'], (list, np.ndarray)) else global_velocity['context'])

In [ ]:
# 'global_angular_velocity': dict with sub-keys ['arr', 'context']
global_angular_velocity = payload['global_angular_velocity']
print("Key 'global_angular_velocity': sub-keys =", list(global_angular_velocity.keys()))
print("  First element of 'global_angular_velocity[\"arr\"]':", global_angular_velocity['arr'][0])
print("  First element of 'global_angular_velocity[\"context\"]':", global_angular_velocity['context'][0] if isinstance(global_angular_velocity['context'], (list, np.ndarray)) else global_angular_velocity['context'])

In [ ]:
# 'skeleton_tree': dict with sub-keys ['node_names', 'parent_indices', 'local_translation']
skeleton_tree = payload['skeleton_tree']
print("Key 'skeleton_tree': sub-keys =", list(skeleton_tree.keys()))
print("  First element of 'skeleton_tree[\"node_names\"]':", skeleton_tree['node_names'][0])
print("  First element of 'skeleton_tree[\"parent_indices\"]':", skeleton_tree['parent_indices']['arr'])
print("  First element of 'skeleton_tree[\"local_translation\"]':", skeleton_tree['local_translation']['arr'])

In [ ]:

# 'is_local': not a dict (type=bool)
print("Key 'is_local':", payload['is_local'], f"(type={type(payload['is_local']).__name__})")

# 'fps': not a dict (type=float)
print("Key 'fps':", payload['fps'], f"(type={type(payload['fps']).__name__})")

# '__name__': not a dict (type=str)
print("Key '__name__':", payload['__name__'], f"(type={type(payload['__name__']).__name__})")

In [ ]:
# 1) Inventory table
inventory_rows = []
for p in npy_files:
    size = p.stat().st_size
    inventory_rows.append({
        "file": str(p),
        "size_bytes": size,
        "size_mb": round(size / (1024 * 1024), 4),
    })

inventory_df = pd.DataFrame(inventory_rows).sort_values("file").reset_index(drop=True)
inventory_df.to_csv(OUT_DIR / "npy_inventory.csv", index=False)
inventory_df.head(10)

In [ ]:
# 2) Stats table
stats_rows = []
for p in npy_files:
    payload = load_payload(p)
    rot, root_t, gvel, gavel, fps = extract_arrays(payload)

    s = summarize_numeric(rot)
    stats_rows.append({
        "file": str(p),
        "fps": fps,
        "frames": int(rot.shape[0]),
        "joints": int(rot.shape[1]),
        "duration_sec": round(rot.shape[0] / fps, 3),
        "rotation_shape": tuple(rot.shape),
        "root_translation_shape": tuple(root_t.shape),
        "global_velocity_shape": tuple(gvel.shape),
        "global_angular_velocity_shape": tuple(gavel.shape),
        **s,
    })

stats_df = pd.DataFrame(stats_rows).sort_values("file").reset_index(drop=True)
stats_df.to_csv(OUT_DIR / "npy_stats.csv", index=False)
stats_df.head(10)

In [ ]:
# 3) Frame-0 quaternion table for every file (one concrete frame)
frame0_rows = []
for p in npy_files:
    payload = load_payload(p)
    rot, _, _, _, _ = extract_arrays(payload)
    if rot.shape[0] == 0:
        continue
    frame0 = rot[0]  # [J, 4]
    for j in range(frame0.shape[0]):
        qx, qy, qz, qw = frame0[j]
        frame0_rows.append({
            "file": str(p),
            "frame": 0,
            "joint_index": j,
            "quat_x": float(qx),
            "quat_y": float(qy),
            "quat_z": float(qz),
            "quat_w": float(qw),
        })

frame0_df = pd.DataFrame(frame0_rows)
frame0_df.to_csv(OUT_DIR / "npy_frame0_rotation.csv", index=False)
frame0_df.head(30)

In [ ]:
# 4) Inspect a few arbitrary frames from one file
file_index = 0
frames_to_show = [0, 1, 2]

path = npy_files[file_index]
payload = load_payload(path)
rot, root_t, _, _, fps = extract_arrays(payload)

print("file:", path)
print("fps:", fps)
print("rotation shape:", rot.shape)
print("root_translation shape:", root_t.shape)

rows = []
for f in frames_to_show:
    if 0 <= f < rot.shape[0]:
        for j in range(rot.shape[1]):
            q = rot[f, j]
            rows.append({
                "frame": f,
                "joint_index": j,
                "quat_x": float(q[0]),
                "quat_y": float(q[1]),
                "quat_z": float(q[2]),
                "quat_w": float(q[3]),
            })

preview_quat_df = pd.DataFrame(rows)
preview_quat_df.head(45)

In [ ]:
# 5) Root translation for same selected frames
root_rows = []
for f in frames_to_show:
    if 0 <= f < root_t.shape[0]:
        x, y, z = root_t[f]
        root_rows.append({
            "frame": f,
            "root_x": float(x),
            "root_y": float(y),
            "root_z": float(z),
        })

root_preview_df = pd.DataFrame(root_rows)
root_preview_df